In [0]:
%sql
CREATE OR REPLACE VIEW cw_ucc_services_archer_amr_cop_mckesson_d.control.vw_file_ingestion_monitor AS
WITH latest_logs AS (
    SELECT
        run_id,
        config_id,
        status,
        row_count,
        start_time,
        end_time,
        error_message,
        created_at,
        ROW_NUMBER() OVER (
            PARTITION BY config_id
            ORDER BY created_at DESC
        ) AS rn
    FROM cw_ucc_services_archer_amr_cop_mckesson_d.control.file_ingestion_log
)
SELECT
    c.id AS config_id,
    c.source_file_name,
    c.sheet_name,
    c.file_type,
    c.target_catalog,
    c.target_schema,
    c.target_table,
    c.pipeline_name,
    c.source_system,
    c.load_frequency,
    c.ingestion_strategy,
    c.is_active,
    c.load_order,
    l.run_id,
    l.status,
    l.row_count,
    l.start_time,
    l.end_time,
    l.error_message,
    l.created_at AS last_run_created_at
FROM cw_ucc_services_archer_amr_cop_mckesson_d.control.file_ingestion_config c
LEFT JOIN latest_logs l
    ON c.id = l.config_id
   AND l.rn = 1;